In [ ]:
%pip install openai python-dotenv

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

In [3]:
env_path = Path.cwd() / ".env"

if not env_path.exists():
    env_path = Path.cwd().parent / "open_ai" / ".env"

print("Using .env:", env_path)

load_dotenv(env_path)

Using .env: /Users/muhammadali/Documents/python/open_ai/.env


True

In [4]:
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("API key loaded successfully ✅")
else:
    print("API key NOT found ❌")

API key loaded successfully ✅


In [5]:
client = OpenAI(api_key=api_key)

print("OpenAI client created successfully ✅")

OpenAI client created successfully ✅


In [7]:

response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": "Hello, how are you?"
            }
        ]
    )

print(response.choices[0].message.content)


Hello! I'm just a computer program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?


In [12]:
# Zero-shot prompt
Content = """
Helo, Your name is alex, you are a helpful assistant.
You'll answer questions related only to Maths only,
if user ask questions related to any other topic, just say
"I am sorry, I can only answer questions related to Maths."
"""
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": Content
        },
        {
            "role": "user",
            "content": "What is the square root of 16?"
        }
    ]
)

print(response.choices[0].message.content)

The square root of 16 is 4.


In [13]:
# Few-shot prompt
Content = """
Helo, Your name is alex, you are a helpful assistant.
You'll answer questions related only to Maths only,
if user ask questions related to any other topic, just say
"I am sorry, I can only answer questions related to Maths.

For Example:
User: What is the square root of 16?
Alex: The square root of 16 is 4.

OR 

User: What is the Capital of France?
Alex: I am sorry, I can only answer questions related to Maths."
"""
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": Content
        },
        {
            "role": "user",
            "content": "What is the square root of 16?"
        }
    ]
)

print(response.choices[0].message.content)

The square root of 16 is 4.


In [ ]:
import json
# Chain of thought prompt
# Automated START -> PLAN -> OUTPUT prompt
content = """
You are a helpful assistant. Respond with exactly one JSON object per turn:
{"step": "start" | "plan" | "output", "content": "text"}

For each request, emit START first, then a brief high-level PLAN, then OUTPUT.
Do not provide hidden chain-of-thought or detailed private reasoning; keep the plan to one concise sentence.
In OUTPUT, give the direct answer with a brief explanation.
"""

question = "What is the square root of 16?"
messages = [
    {"role": "system", "content": content},
    {"role": "user", "content": question}
]

for _ in range(5):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        response_format={"type": "json_object"},
        messages=messages
    )

    answer = response.choices[0].message.content
    result = json.loads(answer)
    print(result)

    messages.append({"role": "assistant", "content": answer})
    if result.get("step", "").lower() == "output":
        break

    messages.append({
        "role": "user",
        "content": "Continue with the next step in the requested START -> PLAN -> OUTPUT sequence."
    })
else:
    print("Stopped: reached the maximum number of steps before OUTPUT.")


{'step': 'start', 'content': 'Starting the calculation.'}
{'step': 'plan', 'content': 'Calculate the square root of the number 16.'}
{'step': 'output', 'content': 'The square root of 16 is 4, because 4 multiplied by 4 equals 16.'}


In [38]:
# CHAIN OF THOUGHT PROMPT

In [67]:
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)


In [ ]:
system_prompt = """Hey you are an expert AI assistant,
in resolving user queries using chain of thought.
You work on START, PLAN and OUTPUT steps.
You need to first plan what need to be done. The plan can be multiple steps. After planning, you need to provide the output.
Once you think enough plan has been done, finally you can give an OUTPUT.

RULES:
- Strictly follow the given JSON output format
- Only run one step at a time. Do not run multiple steps in one go.
- The sequence of step is 
START (Where user gives an input query)
PLAN (That can be multiple times/steps)
OUTPUT (Which is going to be displayed to the user)

OUTPUT JSON FORMAT:
{"step": "start" | "plan" | "output", "content": "string"}

Example:
START: Can you solve 2 + 5 * 5/ 5
PLAN: {"step": "plan", "content": "Seems like user is interested in maths problem."}
PLAN: {"step": "plan", "content": "Looking at problem we should solve using BODMAS rule.:}
PLAN: {"step": "plan", "content": "First we need to solve 5 * 5 = 25"}
PLAN: {"step": "plan", "content": "Then we need to solve 25 / 5 = 5"}
PLAN: {"step": "plan", "content": "Finally we need to solve 2 + 5 = 7"}
OUTPUT: {"step": "output", "content": "7"}
"""

In [ ]:

import json

message_history = [
    {"role": "system", "content": system_prompt}
]

user_query = input("User Input: ")
message_history.append({"role": "user", "content": user_query})

while True:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        response_format={"type": "json_object"},
        messages=message_history
    )

    raw_result = response.choices[0].message.content

     # Save the assistant's response in conversation history
    message_history.append({
            "role": "assistant",
            "content": raw_result
        })
    
    # Convert JSON string into a Python dictionary
    parsed_result = json.loads(raw_result)


    step = parsed_result.get("step", "").lower()
    content = parsed_result.get("content", "")

    if step == "start":
        print("Start:", content)
        continue

    elif step == "plan":
        print("Plan:", content)
        continue

    elif step == "output":
        print("Output:", content)
        break

    else:
        print("Unknown step:", step)
        break


Start: User is asking for a mathematical calculation involving addition, multiplication, and division.
Plan: We will apply the BODMAS rule to the expression 2 + 3 * 4 / 2.
Plan: First, we need to solve the multiplication: 3 * 4 = 12.
Plan: Next, we perform the division: 12 / 2 = 6.
Plan: Finally, we need to perform the addition: 2 + 6 = 8.
Output: 8


In [ ]:
#Manual example of previous code with while loop
response = client.chat.completions.create(
    model="gpt-4o-mini",
    response_format={"type": "json_object"},
    messages=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": "Write a JavaScript function to add n numbers and explain it briefly."
        },
        {
            "role": "assistant",
            "content": json.dumps({
                "step": "START",
                "content": "The user wants a JavaScript function to add n numbers and a brief explanation."
            })
        },
        {
            "role": "assistant",
            "content": json.dumps({
                "step": "PLAN",
                "content": "Create a function that accepts multiple numbers, adds them, and briefly explains how it works."
            })
        }
    ]
)



In [71]:
print(response.choices[0].message.content)

{"step": "PLAN", "content": "The function should be able to accept any number of arguments, so using the rest parameter syntax '...' will be useful."}
